In [18]:
from dotenv import load_dotenv

In [41]:
load_dotenv(override=True)

True

In [58]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# STEP 1
## 1.1 Loading Documents

In [7]:
video_id = "iys_pmJSp9M" # only the ID, not full URL
try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id)

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

Luke here with the Out Boys YouTube channel and welcome to the interior of Alaska I'm going to be camping in the snow without a tent for the next 3 days I'll be building survival shelters sleeping under animal hides making Bushcraft projects next to the fire and cooking amazing food [Music] well we've had a really weird winter here in Alaska and it's been cold then hot then cold again but the good news is the snow's not too deep and the swamps are all Frozen so I can really get out and explore places I otherwise couldn't access [Music] I got myself pretty wet looks like there's a little ditch or creek here so I stopped to get out and walk it before I tried to take the K truck into it and there's a booby trap right here look at that is just snow floating on water then I walked onto it and sunk into mud up to here I think I'll just go around been driving around for about 2 and 1/2 hours and I think it's about time to look for a place to [Applause] Camp yeah there we go look around for so

In [9]:
# transcript_list

## 1.2 Text Splitting

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [12]:
len(chunks)

13

In [14]:
chunks[1]

Document(metadata={}, page_content="sunk into mud up to here I think I'll just go around been driving around for about 2 and 1/2 hours and I think it's about time to look for a place to [Applause] Camp yeah there we go look around for some dead trees it's going to be Pitch Black by 400 p.m. today I have about an hour and a half before Sunset and it's really hard to find dead trees in the dark [Music] there's an invasive Beetle called the Japanese Spruce bark Beetle it's wiping out a lot of the spruce trees in this area so a lot of the trees are sick or dead in the summertime this place is a massive forest fire hazard so burning up the dead trees in the winter time can actually be a real good thing seems like a lot of wood but it's not if you want to keep warm for 18 hours of Darkness Plus have a shelter it takes a lot Sun's setting I got to bust out my lights I use these bicycle lights that have a GoPro Mount I got this on a trip to Kyoto when Tommy was just little [Music] got to get a

## 1.3 Embedding Generation and Storing in Vector Store

In [25]:
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
db = FAISS.from_documents(chunks, embedding)

In [29]:
db.index_to_docstore_id

{0: '6505d0d0-c0b8-4213-bc3e-6480cab79e8c',
 1: '356ad969-e5b0-4c54-85e4-0005ffb7a828',
 2: '6529aa69-b55e-4199-aee3-feb4f793d17f',
 3: '10aa337e-d4c8-40e0-8bfe-180de3b203b6',
 4: '00f774a0-28e2-4cae-b155-298d946d03d2',
 5: 'bffd3e34-c051-41ee-aa68-eee786113015',
 6: '6c6bc324-423f-425f-9201-d3a61cff47fb',
 7: '3964ea5b-42e2-405f-9be5-de133d570206',
 8: '82a2327a-c392-4f1a-bb02-7544f8ac5f1e',
 9: '17c47f42-ef65-4664-91a5-ea47bda3c8a2',
 10: 'be9f13ec-b570-44ea-9b05-19dec6aa0801',
 11: '2aa2bd79-01e3-48dc-a588-33cb8672dec6',
 12: '275b914e-6bd4-4113-a63d-d63ea6588ab1'}

In [32]:
# db.get_by_ids(['356ad969-e5b0-4c54-85e4-0005ffb7a828'])

# STEP 2 - Retriever

In [34]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [35]:
retriever.invoke('Where is Luke camping in the video?')

[Document(id='6505d0d0-c0b8-4213-bc3e-6480cab79e8c', metadata={}, page_content="Luke here with the Out Boys YouTube channel and welcome to the interior of Alaska I'm going to be camping in the snow without a tent for the next 3 days I'll be building survival shelters sleeping under animal hides making Bushcraft projects next to the fire and cooking amazing food [Music] well we've had a really weird winter here in Alaska and it's been cold then hot then cold again but the good news is the snow's not too deep and the swamps are all Frozen so I can really get out and explore places I otherwise couldn't access [Music] I got myself pretty wet looks like there's a little ditch or creek here so I stopped to get out and walk it before I tried to take the K truck into it and there's a booby trap right here look at that is just snow floating on water then I walked onto it and sunk into mud up to here I think I'll just go around been driving around for about 2 and 1/2 hours and I think it's about

# STEP 3 - Augmentation

In [42]:
llm = ChatGroq(model='llama-3.3-70b-versatile')

In [44]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [54]:
question          = "Where is Luke camping in the video?"
question          = "How long is Luke camping without a tent?"
retrieved_docs    = retriever.invoke(question)

In [46]:
# retrieved_docs

[Document(id='6505d0d0-c0b8-4213-bc3e-6480cab79e8c', metadata={}, page_content="Luke here with the Out Boys YouTube channel and welcome to the interior of Alaska I'm going to be camping in the snow without a tent for the next 3 days I'll be building survival shelters sleeping under animal hides making Bushcraft projects next to the fire and cooking amazing food [Music] well we've had a really weird winter here in Alaska and it's been cold then hot then cold again but the good news is the snow's not too deep and the swamps are all Frozen so I can really get out and explore places I otherwise couldn't access [Music] I got myself pretty wet looks like there's a little ditch or creek here so I stopped to get out and walk it before I tried to take the K truck into it and there's a booby trap right here look at that is just snow floating on water then I walked onto it and sunk into mud up to here I think I'll just go around been driving around for about 2 and 1/2 hours and I think it's about

In [51]:
# merging
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"Luke here with the Out Boys YouTube channel and welcome to the interior of Alaska I'm going to be camping in the snow without a tent for the next 3 days I'll be building survival shelters sleeping under animal hides making Bushcraft projects next to the fire and cooking amazing food [Music] well we've had a really weird winter here in Alaska and it's been cold then hot then cold again but the good news is the snow's not too deep and the swamps are all Frozen so I can really get out and explore places I otherwise couldn't access [Music] I got myself pretty wet looks like there's a little ditch or creek here so I stopped to get out and walk it before I tried to take the K truck into it and there's a booby trap right here look at that is just snow floating on water then I walked onto it and sunk into mud up to here I think I'll just go around been driving around for about 2 and 1/2 hours and I think it's about time to look for a place to [Applause] Camp yeah there we go look around for\n

In [55]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

# Step 4 - Generation

In [56]:
answer = llm.invoke(final_prompt)
print(answer.content)

Luke is camping without a tent for 3 days.


# Chain Building

In [59]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [60]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [66]:
# parallel_chain.invoke(question)

In [62]:
parser = StrOutputParser()

In [65]:
chain = parallel_chain | prompt | llm | parser

In [70]:
from IPython.display import Markdown, display
def printmd(string):
    display(Markdown(string))

In [71]:
question = 'can you summarize the video.'
ans = chain.invoke(question)

In [72]:
printmd(ans)

The video appears to be about a person's wilderness survival experience. They are in a cold environment and have set up a camp with a fire. They discuss their dinner, which is moose fajitas, and the importance of having a wind break to keep the smoke out of their face. They also talk about the need to dry out their boots and clothes, and how they will have to stoke the fire throughout the night to stay warm. The next morning, they wake up and make breakfast, and then venture out to find more firewood and a place to camp. They collect dead trees and build a shelter, and reflect on the challenges of surviving in the wilderness. Overall, the video seems to be a documentation of the person's daily life and struggles while living in the wilderness.